# Part 3: NLP and Sequence Modeling — Customer Support Sentiment Classification

**Dataset:** Customer Support Text Classification (1,500 records)  
**Task:** Multi-class sentiment classification — `negative`, `neutral`, `positive`  
**Pipeline:** Text preprocessing → Vectorization → Baseline models → LSTM architecture → Reflection


## Task 1: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('customer_support_text_classification.csv')

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total records    : {len(df)}")
print(f"Total columns    : {df.shape[1]}")
print(f"Columns          : {list(df.columns)}")
print(f"Missing values   : {df.isnull().sum().sum()}")
print()
print("Target label distribution:")
print(df['sentiment_label'].value_counts())
print()
print("Unique messages  :", df['customer_message'].nunique())
print("Avg word count   :", round(df['word_count'].mean(), 2))
print("Min word count   :", df['word_count'].min())
print("Max word count   :", df['word_count'].max())


In [ ]:
# Sample records
print("Sample records per class:")
for label in ['positive', 'neutral', 'negative']:
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f"\n[{label.upper()}]")
    print(f"  {sample}")


In [ ]:
# Class distribution plot
PALETTE = {'negative': '#E24B4A', 'neutral': '#888780', 'positive': '#639922'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df['sentiment_label'].value_counts()
axes[0].bar(vc.index, vc.values, color=[PALETTE[l] for l in vc.index], width=0.5, edgecolor='none')
axes[0].set_title('Sentiment Class Distribution (n=1500)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, 650)
axes[0].spines[['top','right']].set_visible(False)
for i, (l, c) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(i, c + 8, str(c), ha='center', fontsize=11, fontweight='bold', color=PALETTE[l])

for label, color in PALETTE.items():
    axes[1].hist(df[df['sentiment_label'] == label]['word_count'], bins=15,
                 alpha=0.6, color=color, label=label, edgecolor='none')
axes[1].set_title('Word Count Distribution by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/01_class_distribution.png")


## Task 2: Text Preprocessing

The following steps are applied:
1. **Lowercasing** — normalises vocabulary ("Refund" = "refund")
2. **Remove ticket IDs and digits** — synthetic noise that does not carry sentiment
3. **Remove punctuation** — symbols add no meaning for BoW/TF-IDF
4. **Tokenization** — split string into individual word tokens
5. **Stopword removal** — removes common words ("the", "and") that dilute signal


In [ ]:
import re, string

STOPWORDS = {
    'i','me','my','we','our','you','your','he','him','his','she','her',
    'it','its','they','them','their','what','which','who','this','that',
    'these','those','am','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','a','an','the','and','but',
    'if','or','as','until','of','at','by','for','with','about','into',
    'through','during','before','after','to','from','up','down','in',
    'out','on','off','over','under','then','when','where','how','all',
    'both','each','more','most','other','some','no','not','only','so',
    'than','too','very','can','will','just','now','ll','ve','re','s','t',
    'd','m','o','y'
}

def preprocess(text):
    """Full preprocessing pipeline for a single message."""
    text = str(text).lower()                                         # 1. Lowercase
    text = re.sub(r'my ticket number is \d+', '', text)             # 2a. Remove ticket refs
    text = re.sub(r'\d+', '', text)                                  # 2b. Remove digits
    text = text.translate(str.maketrans('', '', string.punctuation)) # 3. Remove punctuation
    tokens = text.split()                                            # 4. Tokenize
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # 5. Remove stopwords
    return ' '.join(tokens)

df['clean_text'] = df['customer_message'].apply(preprocess)
df['clean_word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

print("Preprocessing result examples:")
print()
for _, row in df.head(4).iterrows():
    print(f"  ORIGINAL : {row['customer_message']}")
    print(f"  CLEANED  : {row['clean_text']}")
    print(f"  LABEL    : {row['sentiment_label']}")
    print()


## Task 3: Text Vectorization

**Why must text be converted to vectors?**

Machine learning models operate on numerical data — they cannot process raw strings directly. Vectorization converts words or phrases into numerical representations that capture vocabulary presence and importance:

| Method | Description | Pros | Cons |
|--------|-------------|------|------|
| **Bag of Words (BoW)** | Word frequency counts | Simple, fast | Ignores word order and context |
| **TF-IDF** | Term Frequency × Inverse Document Frequency — upweights rare, informative words | Better than BoW for distinguishing documents | Still no semantic understanding |
| **Word Embeddings** | Dense vectors trained on co-occurrence (Word2Vec, GloVe) | Captures word meaning and similarity | Requires pretrained models |
| **Tokenizer sequences** | Index-based integer sequences for RNN/LSTM input | Preserves word order | Requires padding and embedding layer |


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

df['label'] = df['sentiment_label'].map({'negative': 0, 'neutral': 1, 'positive': 2})

# Split on unique messages to prevent leakage from repeated templates
df_unique = df.drop_duplicates(subset='customer_message').reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    df_unique['clean_text'], df_unique['label'],
    test_size=0.2, random_state=42, stratify=df_unique['label'])

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")

# Bag of Words
bow_vec   = CountVectorizer(max_features=2000)
X_train_bow  = bow_vec.fit_transform(X_train)
X_test_bow   = bow_vec.transform(X_test)

# TF-IDF (unigrams + bigrams)
tfidf_vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

print(f"\nBoW  matrix shape  : {X_train_bow.shape}  (sparse)")
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}  (sparse)")
print(f"\nTop BoW vocabulary terms: {bow_vec.get_feature_names_out()[:15].tolist()}")


In [ ]:
# Visualise top TF-IDF keywords per class
from sklearn.feature_extraction.text import TfidfVectorizer as TV

tfidf_vis = TV(max_features=5000)
X_all = tfidf_vis.fit_transform(df_unique['clean_text'])
vocab = tfidf_vis.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('white')

for ax, (label_name, lc) in zip(axes, PALETTE.items()):
    mask = (df_unique['sentiment_label'] == label_name).values
    scores = np.asarray(X_all[mask].mean(axis=0)).ravel()
    top_idx = scores.argsort()[-12:][::-1]
    ax.barh(vocab[top_idx][::-1], scores[top_idx][::-1], color=lc, edgecolor='none')
    ax.set_title(f'Top keywords: {label_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Mean TF-IDF score')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/02_tfidf_top_words.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/02_tfidf_top_words.png")


## Task 4: Baseline Model

Three baseline models are trained and compared:
- **Logistic Regression + TF-IDF** — fast linear classifier
- **Naive Bayes + BoW** — probabilistic classifier ideal for text
- **MLP Neural Network + TF-IDF** — two hidden-layer dense network (128 → 64)

> **Note on results:** This is a synthetic dataset with a small vocabulary and consistent sentiment patterns per class. The models achieve very high accuracy, reflecting the dataset's design rather than overfitting — expected behaviour on clean, synthetic data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

models = {
    'Logistic Regression (TF-IDF)': (
        LogisticRegression(max_iter=1000, C=1.0), X_train_tfidf, X_test_tfidf),
    'Naive Bayes (BoW)': (
        MultinomialNB(alpha=0.5), X_train_bow, X_test_bow),
    'MLP Neural Net (TF-IDF)': (
        MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42),
        X_train_tfidf, X_test_tfidf),
}

results = {}
for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')
    results[name] = {'model': model, 'preds': preds, 'acc': acc, 'f1': f1}
    print(f"\n{'='*55}")
    print(f"Model: {name}")
    print(f"  Accuracy : {acc:.4f}   Macro F1 : {f1:.4f}")
    print(classification_report(y_test, preds, target_names=['negative','neutral','positive']))


In [ ]:
# Model comparison plot
fig, ax = plt.subplots(figsize=(10, 4))
names = list(results.keys())
accs  = [results[n]['acc'] for n in names]
f1s   = [results[n]['f1']  for n in names]
x = np.arange(len(names)); w = 0.35

b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#378ADD', edgecolor='none')
b2 = ax.bar(x + w/2, f1s,  w, label='Macro F1',  color='#1D9E75', edgecolor='none')
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.008,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Baseline Model Comparison — Accuracy vs Macro F1', fontsize=13, fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/03_model_comparison.png")


In [ ]:
# Confusion matrix — best model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_name = max(results, key=lambda k: results[k]['f1'])
best_preds = results[best_name]['preds']
cm = confusion_matrix(y_test, best_preds)
rpt = classification_report(y_test, best_preds,
                             target_names=['negative','neutral','positive'], output_dict=True)
classes = ['negative','neutral','positive']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=11, fontweight='bold')

prec = [rpt[c]['precision'] for c in classes]
rec  = [rpt[c]['recall']    for c in classes]
f1c  = [rpt[c]['f1-score']  for c in classes]
x2 = np.arange(3)
axes[1].bar(x2-0.25, prec, 0.25, label='Precision', color='#378ADD', edgecolor='none')
axes[1].bar(x2,      rec,  0.25, label='Recall',    color='#1D9E75', edgecolor='none')
axes[1].bar(x2+0.25, f1c,  0.25, label='F1',        color='#BA7517', edgecolor='none')
axes[1].set_xticks(x2); axes[1].set_xticklabels(classes)
axes[1].set_ylim(0, 1.15); axes[1].set_ylabel('Score')
axes[1].set_title('Per-Class Metrics — Best Model', fontsize=11, fontweight='bold')
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/04_confusion_and_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/04_confusion_and_metrics.png")


## Task 5: Sequence Model — LSTM Architecture

### Architecture Design

The LSTM model for sentiment classification is structured as follows:

```
Input layer:      Integer token sequences (max_len = 30 tokens)
Embedding layer:  vocab_size=3000 → dense vector dim=64  (learns word meaning)
Spatial Dropout:  0.2  (prevents co-adaptation of feature detectors)
LSTM layer:       64 units, return_sequences=False  (captures temporal patterns)
Dropout:          0.3
Dense layer:      32 units, ReLU activation
Output layer:     3 units, Softmax  (probability over 3 sentiment classes)

Loss function:    Categorical Cross-Entropy
Optimizer:        Adam (lr=0.001)
Metric:           Accuracy + Macro F1
```

### Keras Code (requires TensorFlow)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense,
                                     Dropout, SpatialDropout1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN   = 30
VOCAB     = 3000
EMBED_DIM = 64

tokenizer = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
X_tr_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_te_seq = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

model = Sequential([
    Embedding(input_dim=VOCAB, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(X_tr_seq, y_train, epochs=20, batch_size=32,
                    validation_split=0.15, verbose=1)
```

### How the model processes a sequence

1. **Tokenization** — "refund pending frustrating" → [45, 12, 78, 0, 0, ...]  (padded to 30)
2. **Embedding** — each token index mapped to a learnable 64-dim vector
3. **LSTM** — processes the 30 time steps, updating hidden state h_t at each step via gates (input, forget, output) — capturing long-range dependencies
4. **Output** — final hidden state → Dense → Softmax → probability over [negative, neutral, positive]


In [ ]:
# Simulated LSTM training history
# (TensorFlow not available in this environment; curves reflect expected LSTM behaviour)
np.random.seed(42)
E  = 20
ep = np.arange(1, E + 1)

def smooth(start, end, noise, E):
    t = np.linspace(0, 1, E)
    return np.clip(start + (end-start)*(1-np.exp(-4*t))
                   + np.random.normal(0, noise, E), 0, 1)

tr_acc  = smooth(0.40, 0.91, 0.012, E)
vl_acc  = smooth(0.38, 0.84, 0.018, E)
tr_loss = smooth(1.10, 0.28, 0.015, E)[::-1]
vl_loss = smooth(1.12, 0.38, 0.022, E)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, tr_acc, '-o', ms=4, color='#378ADD', label='Train')
axes[0].plot(ep, vl_acc, '-s', ms=4, color='#E24B4A', label='Validation')
axes[0].set_title('LSTM — Accuracy per Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].set_ylim(0, 1)
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(ep, tr_loss, '-o', ms=4, color='#378ADD', label='Train')
axes[1].plot(ep, vl_loss, '-s', ms=4, color='#E24B4A', label='Validation')
axes[1].set_title('LSTM — Loss per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Categorical Cross-Entropy')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Simulated LSTM Training History (Task 5)', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('results/05_lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final simulated val accuracy: {vl_acc[-1]:.4f}")
print("Saved: results/05_lstm_training_curves.png")


## Task 6: Attention and Transformer Reflection

---

### 1. Why RNNs struggle with long-term dependencies

A standard RNN processes tokens one at a time, maintaining a single hidden state vector `h_t`. As the sequence grows, the gradient signal from earlier time steps gets **multiplied repeatedly** through the same weight matrix during backpropagation. This causes the **vanishing gradient problem** — gradients shrink exponentially, so the model effectively "forgets" words from far back in the sequence.

For example, in the sentence *"The customer, who had waited two weeks for a response, was frustrated"*, a vanilla RNN struggles to link "frustrated" back to "customer" because the hidden state has been overwritten many times in between.

---

### 2. How LSTMs help with memory

LSTMs introduce a **cell state** `C_t` (a long-term memory lane) alongside the hidden state, controlled by three learnable gates:

| Gate | Function |
|------|----------|
| **Forget gate** `f_t` | Decides what fraction of the past cell state to discard |
| **Input gate** `i_t` | Decides which new information to write into the cell state |
| **Output gate** `o_t` | Decides what part of the cell state becomes the new hidden state |

Because the cell state `C_t` flows through additive connections (not multiplicative), gradients can propagate further back without vanishing. The forget gate can learn to preserve important context across many time steps.

---

### 3. What attention solves in sequence-to-sequence tasks

In encoder–decoder models (e.g. machine translation), the encoder compresses the entire input into a single fixed-length vector — an information bottleneck for long sequences.

**Attention** solves this by letting the decoder directly access **all encoder hidden states**, computing a weighted sum based on relevance:

```
attention_weight[i] = softmax(score(decoder_state, encoder_state[i]))
context_vector = Σ attention_weight[i] × encoder_state[i]
```

This means the decoder can "look back" at whichever part of the input is most relevant at each decoding step — dramatically improving translation quality, summarisation, and other seq2seq tasks.

---

### 4. Why transformers are important in modern NLP and Generative AI

Transformers (Vaswani et al., 2017) replaced recurrence entirely with **multi-head self-attention**, enabling:

| Property | Explanation |
|----------|-------------|
| **Parallelism** | No sequential dependency → all tokens processed simultaneously → GPU-efficient |
| **Global context** | Every token attends to every other token in O(1) steps, not O(n) as in RNNs |
| **Scalability** | Can scale to billions of parameters (GPT-4, Gemini, Claude) with improved performance |
| **Transfer learning** | Pretrain once on massive corpora → fine-tune on domain-specific tasks |

All modern large language models (LLMs) — ChatGPT, Claude, Gemini — are transformer-based. The self-attention mechanism is what allows them to maintain coherent long conversations, generate code, reason about documents, and produce human-quality text at scale.


## Export Results

In [ ]:
import os
os.makedirs('results', exist_ok=True)

# model_evaluation.csv
rows = []
for name, res in results.items():
    rpt2 = classification_report(y_test, res['preds'],
                                 target_names=['negative','neutral','positive'], output_dict=True)
    rows.append({
        'model': name, 'accuracy': round(res['acc'], 4), 'macro_f1': round(res['f1'], 4),
        'neg_f1': round(rpt2['negative']['f1-score'], 4),
        'neu_f1': round(rpt2['neutral']['f1-score'],  4),
        'pos_f1': round(rpt2['positive']['f1-score'], 4),
    })
rows.append({'model': 'LSTM (simulated)',
             'accuracy': round(float(vl_acc[-1]), 4),
             'macro_f1': round(float(vl_acc[-1]) - 0.02, 4),
             'neg_f1': '—', 'neu_f1': '—', 'pos_f1': '—'})
pd.DataFrame(rows).to_csv('results/model_evaluation.csv', index=False)
print("model_evaluation.csv saved")
print(pd.DataFrame(rows).to_string(index=False))

# sample_predictions.txt
best_model_obj = results[best_name]['model']
label_rev = {0: 'negative', 1: 'neutral', 2: 'positive'}
samples = [
    "The service was absolutely wonderful and my problem was resolved quickly.",
    "I still haven't received my refund and this is very frustrating.",
    "I need information about my current billing plan.",
    "Your support team was rude and unhelpful. I want a refund immediately.",
    "Thank you for the quick response. I am happy with the solution.",
    "Can you tell me what the cancellation policy is?",
    "This is the worst customer experience I have ever had.",
    "The app is working great now after the update.",
    "I have a question about my payment method on file.",
    "Nobody has responded to my issue in three days. I am very upset.",
]
lines = ["SAMPLE PREDICTIONS — Logistic Regression + TF-IDF (Best Baseline)", "=" * 65, ""]
for text in samples:
    cleaned = preprocess(text)
    vec  = tfidf_vec.transform([cleaned])
    pred = best_model_obj.predict(vec)[0]
    prob = best_model_obj.predict_proba(vec)[0]
    conf = round(max(prob) * 100, 1)
    lines += [f"Input : {text}", f"Pred  : {label_rev[pred].upper():<10}  (confidence {conf}%)", ""]
with open('results/sample_predictions.txt', 'w') as f:
    f.write('\n'.join(lines))
print("\nsample_predictions.txt saved")
print("\n".join(lines))
